# CA2 DATA VISUALISATION: PART 2
Chan Shuo Kai (P2300533) DAAA/FT/2A/21

Plotting using plotly and creating dashboards using dash. This file contains 2 plots and 2 dashboards.

### IMPORT MODULES AND DATASETS

In [1]:
import os
import pandas as pd
import numpy as np

import sys
import dash
from dash import dcc, html, Input, Output
import plotly.express as px
import plotly.graph_objects as go
import dash_bootstrap_components as dbc


os.chdir(r"C:\SP\Year 2 Sem 2\DAVI\P2300533-ChanShuoKai-Part2\dataset")

daily_ridership = pd.read_csv('cleaned_yearly_ave_daily_ridership.csv')
mrt_fares = pd.read_csv('FaresforMRTandLRTCentsEffectivefrom28December2024.csv')
trip_distance = pd.read_csv('PublicTransportUtilisationAverageTripDistance.csv')

### STANDALONE PLOTLY CHARTS
#### PLOTLY CHART 1: SANKEY

In [2]:
# Categorize trips
trip_distance["distance_category"] = trip_distance["ave_distance_per_trip"].apply(
    lambda x: "Long Distance" if x > 5 else "Short Distance"
)

trip_distance["distance_bucket"] = trip_distance["ave_distance_per_trip"].apply(
    lambda x: "0-2 km" if x < 2 else ("2-5 km" if x < 5 else ("5-8 km" if x < 8 else "8+ km"))
)

In [3]:
# Aggregate flows
flow_mode_category = trip_distance.groupby(["mode", "distance_category"]).size().reset_index(name="count")
flow_category_bucket = trip_distance.groupby(["distance_category", "distance_bucket"]).size().reset_index(name="count")


# Prepare node labels
modes = list(trip_distance["mode"].unique())
distance_categories = list(trip_distance["distance_category"].unique())
bucket_order = ["0-2 km", "2-5 km", "5-8 km", "8+ km"]
distance_buckets = [b for b in bucket_order if b in trip_distance["distance_bucket"].unique()]

labels = modes + distance_categories + distance_buckets
label_to_index = {label: i for i, label in enumerate(labels)}


# Build link data
sources = (
    flow_mode_category["mode"].map(label_to_index).tolist() +
    flow_category_bucket["distance_category"].map(label_to_index).tolist()
)
targets = (
    flow_mode_category["distance_category"].map(label_to_index).tolist() +
    flow_category_bucket["distance_bucket"].map(label_to_index).tolist()
)
values = flow_mode_category["count"].tolist() + flow_category_bucket["count"].tolist()


# Define colors
mode_colors = px.colors.qualitative.Plotly
node_colors = [mode_colors[i % len(mode_colors)] for i in range(len(modes))]

cat_color_map = {"Short Distance": "#a6cee3", "Long Distance": "#1f78b4"}
node_colors += [cat_color_map.get(cat, "#cccccc") for cat in distance_categories]

bucket_color_map = {"0-2 km": "#b2df8a", "2-5 km": "#33a02c", "5-8 km": "#fb9a99", "8+ km": "#e31a1c"}
node_colors += [bucket_color_map.get(bucket, "#cccccc") for bucket in distance_buckets]

link_colors = [node_colors[s] for s in sources]


# Define fixed node positions
num_modes, num_cats, num_buckets = len(modes), len(distance_categories), len(distance_buckets)
node_x = [0.0] * num_modes + [0.35] * num_cats + [0.6] * num_buckets
node_y = ([i/(num_modes+1) for i in range(1, num_modes+1)] +
          [i/(num_cats+1) for i in range(1, num_cats+1)] +
          [i/(num_buckets+1) for i in range(1, num_buckets+1)])


fig = go.Figure(data=[go.Sankey(
    arrangement="fixed",
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=labels,
        color=node_colors,
        x=node_x,
        y=node_y
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values,
        color=link_colors
    )
)])

fig.update_layout(
    title_text="Sankey Diagram: Transport Modes → Distance Type → Distance Split",
    font_size=10,
    template="plotly_dark"
)
fig.show()


#### PLOTLY CHART 2: VIOIN

In [4]:
fig = px.violin(
    trip_distance,
    x="distance_category",
    y="ave_distance_per_trip",
    title="Distribution of Average Distance per Trip by Category",
    template="plotly_dark",
    category_orders={"distance_category": ["Short Distance", "Long Distance"]},
    labels={"ave_distance_per_trip": "Average Distance per Trip (km)"},
    box=True,           
    points="all",      
    hover_data=trip_distance.columns 
)

fig.update_traces(
    meanline_visible=True,
    marker=dict(opacity=0.6, size=6),
    line=dict(width=2)
)

fig.update_layout(
    title={'x': 0.5, 'xanchor': 'center'},
    xaxis_title="Distance Category",
    yaxis_title="Average Distance per Trip (km)",
    violingap=0.3,         
    violingroupgap=0.5,    
)

fig.show()


### PLOTLY DASHBOARDS PLOTS:

#### DASHBOARD CHART 1: SUNBURST

In [5]:
# def create_sunburst(selected_modes, top_n):
#     filtered = daily_ridership[daily_ridership["mode"].isin(selected_modes)]
#     if filtered.empty:
#         return px.sunburst(title="No data", template="plotly_dark")
    
#     # Get the top n years by total ridership.
#     top_years = (
#         filtered.groupby("year")["ridership"]
#         .sum()
#         .nlargest(top_n)
#         .index
#         .tolist()
#     )
    
#     # Keep only rows for the top years and convert 'year' to a string.
#     filtered = filtered[filtered["year"].isin(top_years)]
#     filtered["year_str"] = filtered["year"].astype(str)
    
#     return px.sunburst(
#         filtered,
#         path=["mode", "year_str"],
#         values="ridership",
#         title=f"Ridership Breakdown (Top {top_n} Years)",
#         template="plotly_dark"
#     )

# # Initialize the Dash app with dark theme
# external_stylesheets = [dbc.themes.DARKLY]
# app = dash.Dash(__name__, external_stylesheets=external_stylesheets)
# app.title = "Ridership Breakdown Sunburst"

# # Define app layout
# app.layout = dbc.Container(
#     [
#         dbc.Row(
#             dbc.Col(
#                 html.H1(
#                     "Ridership Breakdown Sunburst",
#                     className="text-center text-white mb-4"
#                 )
#             )
#         ),
#         dbc.Row(
#             [
#                 dbc.Col(
#                     dcc.Dropdown(
#                         id="mode-dropdown",
#                         options=[{"label": m, "value": m} for m in daily_ridership["mode"].unique()],
#                         value=["MRT", "LRT", "Bus"],
#                         multi=True,
#                         style={'backgroundColor': '#222222', 'color': '#000000'}
#                     ),
#                     width=6
#                 ),
#                 dbc.Col(
#                     dcc.Input(
#                         id="top-n-input",
#                         type="number",
#                         value=3,
#                         min=1,
#                         max=len(daily_ridership["year"].unique()),
#                         style={'backgroundColor': '#222222', 'color': '#FFFFFF', 'width': '100%'}
#                     ),
#                     width=2
#                 ),
#                 dbc.Col(
#                     html.Div(
#                         "Top N Years",
#                         style={'color': '#FFFFFF', 'fontSize': '16px', 'paddingTop': '10px'}
#                     ),
#                     width=2
#                 )
#             ],
#             justify="center",
#             className="mb-4"
#         ),
#         dbc.Row(
#             dbc.Col(
#                 dcc.Graph(id="sunburst-chart"),
#                 width=12
#             )
#         )
#     ],
#     fluid=True,
#     style={"backgroundColor": "#111111", "padding": "20px"}
# )

# # Callback to update the sunburst chart
# @app.callback(
#     Output("sunburst-chart", "figure"),
#     [Input("mode-dropdown", "value"),
#      Input("top-n-input", "value")]
# )
# def update_chart(selected_modes, top_n):
#     if top_n is None or top_n <= 0:
#         top_n = 3  # Default value for top_n

#     return create_sunburst(selected_modes, top_n)

# if __name__ == '__main__':
#     app.run_server(debug=True, port=8051)

#### DASHBOARD CHART 2: BAR CHART

In [6]:
daily_ridership["growth_rate"] = daily_ridership.groupby("mode")["ridership"].pct_change() * 100

fig = px.bar(daily_ridership, x="year", y="growth_rate", color="mode",
             title="Yearly Ridership Growth Rate (%)",
             template="plotly_dark")

for trace in fig.data:
    if trace.name == "LRT":
        trace.visible = "legendonly"

fig.show()


#### DASHBOARD CHART 3: BOX PLOT

In [7]:
# Ensure 'fare_per_ride' is numeric
mrt_fares["fare_per_ride"] = pd.to_numeric(mrt_fares["fare_per_ride"], errors="coerce")


# Function to extract the minimum distance from the distance string
def get_min_distance(s):
    s = s.strip()
    if "Up to" in s:
        return float(s.replace("Up to", "").replace("km", "").strip())
    elif "Over" in s:
        return float(s.replace("Over", "").replace("km", "").strip())
    elif "-" in s:
        return float(s.split("-")[0].replace("km", "").strip())
    else:
        return float(s.replace("km", "").strip())


# Create 'min_distance' 
if "min_distance" not in mrt_fares.columns:
    mrt_fares["min_distance"] = mrt_fares["distance"].apply(get_min_distance)
else:
    mrt_fares["min_distance"] = pd.to_numeric(mrt_fares["min_distance"], errors="coerce")


# Compute 'cost_per_km' 
if "cost_per_km" not in mrt_fares.columns:
    mrt_fares["cost_per_km"] = mrt_fares["fare_per_ride"] / mrt_fares["min_distance"]
else:
    mrt_fares["cost_per_km"] = pd.to_numeric(mrt_fares["cost_per_km"], errors="coerce")


# Create a 'distance_bin' 
if "distance_bin" not in mrt_fares.columns:
    bins = [0, 3.2, 7, 12, np.inf]
    labels = ["(0,3.2]", "(3.2,7]", "(7,12]", "(12,inf)"]
    mrt_fares["distance_bin"] = pd.cut(mrt_fares["min_distance"], bins=bins, labels=labels, right=True)


fig = px.box(
    mrt_fares,
    x="distance_bin",
    y="cost_per_km",
    color="applicable_time",
    points="all",
    title="Cost per KM Distribution by Distance Bin & Applicable Time"
)

fig.update_layout(
    title={'x': 0.5, 'xanchor': 'center'},
    xaxis_title="Trip Distance Range (km)",
    yaxis_title="Fare Cost per Kilometer",
    template="plotly_dark"
)

fig.update_xaxes(
    tickmode='array',
    tickvals=["(0,3.2]", "(3.2,7]", "(7,12]", "(12,inf)"],
    ticktext=["0 to 3.2 km", "3.2 to 7 km", "7 to 12 km", "12+ km"]
)

fig.show()



#### DASHBOARD CHART 4: RIDERSHIP VS DISTANCE BAR

In [8]:
# Standardize mode names: change "Public Bus" in ridership data to "Bus"
daily_ridership['mode'] = daily_ridership['mode'].replace({'Public Bus': 'Bus'})

df = pd.merge(daily_ridership, trip_distance, on=['year', 'mode'], how='inner')
df = df.sort_values(['mode', 'year'])

modes = df['mode'].unique()


fig = go.Figure()

# Create traces for each mode 
all_traces = []
for mode in modes:
    df_mode = df[df['mode'] == mode]
    trace_bar = go.Bar(
        x=df_mode['year'],
        y=df_mode['ridership'],
        name=f"{mode} Daily Ridership",
        yaxis="y1"
    )
    trace_line = go.Scatter(
        x=df_mode['year'],
        y=df_mode['ave_distance_per_trip'],
        mode="lines+markers",
        name=f"{mode} Avg Trip Distance",
        yaxis="y2"
    )
    all_traces.append((mode, trace_bar, trace_line))


# Add traces to the figure; only show the first mode by default
for i, (mode, trace_bar, trace_line) in enumerate(all_traces):
    visible = True if i == 0 else False
    trace_bar.visible = visible
    trace_line.visible = visible
    fig.add_trace(trace_bar)
    fig.add_trace(trace_line)


# Create dropdown menu buttons to update trace visibility per mode
num_traces = len(all_traces) * 2  # 2 traces per mode
buttons = []
for i, (mode, _, _) in enumerate(all_traces):
    vis = [False] * num_traces
    vis[i * 2] = True       # Show bar trace for selected mode
    vis[i * 2 + 1] = True   # Show line trace for selected mode
    buttons.append(dict(
        label=mode,
        method="update",
        args=[{"visible": vis},
              {"title": f"Daily Ridership and Avg Trip Distance for {mode}"}]
    ))


fig.update_layout(
    title=f"Daily Ridership and Avg Trip Distance for {all_traces[0][0]}",
    xaxis=dict(title="Year"),
    yaxis=dict(title="Daily Ridership", side="left"),
    yaxis2=dict(title="Average Trip Distance (km)", overlaying="y", side="right"),
    updatemenus=[dict(
        active=0,
        buttons=buttons,
        x=0.1,
        y=1.15,
        xanchor="left",
        yanchor="top"
    )],
    template="plotly_dark"
)

fig.show()


## PLOTLY DASHBOARD USING DASH

In [9]:
# Data Preparation for Dashboard 1: Ridership Breakdown & Growth
daily_ridership["growth_rate"] = daily_ridership.groupby("mode")["ridership"].pct_change() * 100 # Compute the growth rate (% change) per mode.

df = daily_ridership.copy()

# Get year limits and marks for RangeSlider.
min_year_1 = int(df["year"].min())
max_year_1 = int(df["year"].max())
marks_1 = {year: str(year) for year in range(min_year_1, max_year_1 + 1)}


# Build Dashboard 1 charts
def create_sunburst(selected_modes, top_n, year_range):
    # If top_n is missing or invalid, default to 3
    if top_n is None or not isinstance(top_n, int):
        top_n = 3
    filtered = df[df["mode"].isin(selected_modes)]
    filtered = filtered[(filtered["year"] >= year_range[0]) & (filtered["year"] <= year_range[1])]
    if filtered.empty:
        return px.sunburst(title="No data", template="plotly_dark")
    # Select the top N years (by total ridership)
    top_years = (
        filtered.groupby("year")["ridership"]
        .sum()
        .nlargest(top_n)
        .index
        .tolist()
    )
    filtered = filtered[filtered["year"].isin(top_years)]
    filtered["year_str"] = filtered["year"].astype(str)
    fig = px.sunburst(
        filtered,
        path=["mode", "year_str"],
        values="ridership",
        title=f"Ridership Breakdown (Top {top_n} Years)",
        template="plotly_dark"
    )
    return fig

def create_growth_rate_chart(selected_modes, year_range):
    filtered = df[df["mode"].isin(selected_modes)]
    filtered = filtered[(filtered["year"] >= year_range[0]) & (filtered["year"] <= year_range[1])]
    if filtered.empty:
        return px.bar(title="No data", template="plotly_dark")
    fig = px.bar(
        filtered,
        x="year",
        y="growth_rate",
        color="mode",
        title="Yearly Ridership Growth Rate (%)",
        template="plotly_dark"
    )
    return fig


# Data Preparation for Dashboard 2: Transport Utilisation & Fare Analysis
daily_ridership['mode'] = daily_ridership['mode'].replace({'Public Bus': 'Bus'}) # Standardize mode names: change "Public Bus" to "Bus"

# Merge the datasets on 'year' and 'mode'
df_merged = pd.merge(daily_ridership, trip_distance, on=['year', 'mode'], how='inner')
df_merged = df_merged.sort_values(['mode', 'year'])

modes = sorted(df_merged['mode'].unique())
min_year_2 = int(df_merged['year'].min())
max_year_2 = int(df_merged['year'].max())

# --- Box Plot Data ---
df_fares = pd.read_csv('FaresforMRTandLRTCentsEffectivefrom28December2024.csv')
df_fares["fare_per_ride"] = pd.to_numeric(df_fares["fare_per_ride"], errors="coerce")


# Function to extract the minimum distance from a distance string.
def get_min_distance(s):
    s = s.strip()
    if "Up to" in s:
        return float(s.replace("Up to", "").replace("km", "").strip())
    elif "Over" in s:
        return float(s.replace("Over", "").replace("km", "").strip())
    elif "-" in s:
        return float(s.split("-")[0].replace("km", "").strip())
    else:
        return float(s.replace("km", "").strip())

if "min_distance" not in df_fares.columns:
    df_fares["min_distance"] = df_fares["distance"].apply(get_min_distance)
else:
    df_fares["min_distance"] = pd.to_numeric(df_fares["min_distance"], errors="coerce")

if "cost_per_km" not in df_fares.columns:
    df_fares["cost_per_km"] = df_fares["fare_per_ride"] / df_fares["min_distance"]
else:
    df_fares["cost_per_km"] = pd.to_numeric(df_fares["cost_per_km"], errors="coerce")

if "distance_bin" not in df_fares.columns:
    bins = [0, 3.2, 7, 12, np.inf]
    labels = ["(0,3.2]", "(3.2,7]", "(7,12]", "(12,inf)"]
    df_fares["distance_bin"] = pd.cut(df_fares["min_distance"], bins=bins, labels=labels, right=True)

time_options = sorted(df_fares['applicable_time'].dropna().unique())


# Build the Combined Dashboard Layout (with Tabs)
external_stylesheets = [dbc.themes.DARKLY]
app = dash.Dash(__name__, external_stylesheets=external_stylesheets)
app.title = "Singapore Public Transport Dashboard"

app.layout = dbc.Container([
    dbc.Row([
        dbc.Col(html.H1("Singapore Public Transport Dashboard",
                        className="text-center mb-4"),
                width=12)
    ]),
    dbc.Tabs([
        # ---------------- Tab 1: Dashboard 1 ----------------
        dbc.Tab(label="Ridership Breakdown & Growth", tab_id="tab-1", children=[
            dbc.Card([
                dbc.CardHeader(
                    html.H5("Ridership Breakdown & Growth", className="text-center")
                ),
                dbc.CardBody([
                    dbc.Row([
                        dbc.Col(
                            dcc.RangeSlider(
                                id='year-range-slider-1',
                                min=min_year_1,
                                max=max_year_1,
                                value=[min_year_1, max_year_1],
                                marks={str(year): str(year) for year in range(min_year_1, max_year_1 + 1)},
                                step=1,
                                tooltip={"placement": "bottom", "always_visible": True}
                            ),
                            width=12
                        )
                    ], className="mb-4"),
                    dbc.Row([
                        dbc.Col([
                            html.Label("Select Transport Modes", className="text-white"),
                            dcc.Dropdown(
                                id='mode-dropdown-1',
                                options=[{'label': m, 'value': m} for m in sorted(df['mode'].unique())],
                                value=[sorted(df['mode'].unique())[0]],
                                multi=True,
                                className="mb-3"
                            )
                        ], md=4),
                        dbc.Col([
                            html.Label("Top N Years", className="text-white"),
                            dcc.Input(
                                id='top-n-input-1',
                                type="number",
                                value=3,
                                min=1,
                                max=len(df["year"].unique()),
                                className="mb-3",
                                style={'width': '100%'}
                            )
                        ], md=2)
                    ], justify="center", className="mb-4"),
                    dbc.Row([
                        dbc.Col(html.Div(id='dashboard1-info-cards'), width=12)
                    ], className="mb-4"),
                    dbc.Row([
                        dbc.Col(dcc.Graph(id='sunburst-chart'), width=6),
                        dbc.Col(dcc.Graph(id='growth-rate-chart'), width=6)
                    ], className="mb-5")
                ])
            ])
        ]),
        # ---------------- Tab 2: Dashboard 2 ----------------
        dbc.Tab(label="Transport Utilisation & Fare Analysis", tab_id="tab-2", children=[
            # Dual-Axis Chart Section
            dbc.Row([
                dbc.Col([
                    dbc.Card([
                        dbc.CardHeader(
                            html.H5("Daily Ridership & Avg Trip Distance", className="text-center")
                        ),
                        dbc.CardBody([
                            dbc.Row([
                                dbc.Col([
                                    html.Label("Select Transport Mode", className="text-white"),
                                    dcc.Dropdown(
                                        id='mode-dropdown-2',
                                        options=[{'label': mode, 'value': mode} for mode in modes],
                                        value=modes[0],
                                        clearable=False,
                                        className="mb-3"
                                    )
                                ], md=4),
                                dbc.Col([
                                    html.Label("Select Year Range", className="text-white"),
                                    dcc.RangeSlider(
                                        id='year-range-slider-2',
                                        min=min_year_2,
                                        max=max_year_2,
                                        value=[min_year_2, max_year_2],
                                        marks={str(year): str(year) for year in range(min_year_2, max_year_2 + 1)},
                                        className="mb-3"
                                    )
                                ], md=8)
                            ]),
                            html.Div(id='dual-axis-info-cards'),
                            dcc.Graph(id='dual-axis-chart')
                        ])
                    ])
                ], width=12)
            ], className="mb-5"),
            
            # Box Plot Section
            dbc.Row([
                dbc.Col([
                    dbc.Card([
                        dbc.CardHeader(
                            html.H5("Cost per KM Distribution", className="text-center")
                        ),
                        dbc.CardBody([
                            dbc.Row([
                                dbc.Col([
                                    html.Label("Select Applicable Time", className="text-white"),
                                    dcc.Dropdown(
                                        id='time-dropdown',
                                        options=[{'label': t, 'value': t} for t in time_options],
                                        value=time_options,
                                        multi=True,
                                        className="mb-3"
                                    )
                                ], md=6)
                            ]),
                            html.Div(id='box-plot-info-cards'),
                            dcc.Graph(id='box-plot-chart')
                        ])
                    ])
                ], width=12)
            ])
        ])
    ], id="tabs", active_tab="tab-1")
], fluid=True, style={"backgroundColor": "#111111", "padding": "20px"})


# Callbacks for Dashboard 1 (Ridership Breakdown & Growth)
@app.callback(
    Output('sunburst-chart', 'figure'),
    [Input('mode-dropdown-1', 'value'),
     Input('top-n-input-1', 'value'),
     Input('year-range-slider-1', 'value')]
)
def update_sunburst_chart(selected_modes, top_n, year_range):
    return create_sunburst(selected_modes, top_n, year_range)

@app.callback(
    Output('growth-rate-chart', 'figure'),
    [Input('mode-dropdown-1', 'value'),
     Input('year-range-slider-1', 'value')]
)
def update_growth_rate(selected_modes, year_range):
    return create_growth_rate_chart(selected_modes, year_range)

@app.callback(
    Output('dashboard1-info-cards', 'children'),
    [Input('mode-dropdown-1', 'value'),
     Input('year-range-slider-1', 'value')]
)
def update_info_cards_1(selected_modes, year_range):
    filtered = df[df["mode"].isin(selected_modes)]
    filtered = filtered[(filtered["year"] >= year_range[0]) & (filtered["year"] <= year_range[1])]
    total_ridership = filtered["ridership"].sum()
    avg_growth = filtered["growth_rate"].mean()
    years_included = filtered["year"].nunique()
    info_cards = dbc.Row([
        dbc.Col(dbc.Card([
            dbc.CardHeader("Total Ridership"),
            dbc.CardBody(html.H4(f"{total_ridership:,.0f}", className="card-text"))
        ], color="secondary", inverse=True), md=4),
        dbc.Col(dbc.Card([
            dbc.CardHeader("Average Growth Rate (%)"),
            dbc.CardBody(html.H4(f"{avg_growth:.2f}", className="card-text"))
        ], color="secondary", inverse=True), md=4),
        dbc.Col(dbc.Card([
            dbc.CardHeader("Years Included"),
            dbc.CardBody(html.H4(f"{years_included}", className="card-text"))
        ], color="secondary", inverse=True), md=4)
    ], className="mb-3")
    return info_cards


# Callbacks for Dashboard 2 (Transport Utilisation & Fare Analysis)
@app.callback(
    [Output('dual-axis-chart', 'figure'),
     Output('dual-axis-info-cards', 'children')],
    [Input('mode-dropdown-2', 'value'),
     Input('year-range-slider-2', 'value')]
)
def update_dual_axis_chart(selected_mode, year_range):
    dff = df_merged[(df_merged['mode'] == selected_mode) &
                    (df_merged['year'] >= year_range[0]) &
                    (df_merged['year'] <= year_range[1])]
    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=dff['year'],
        y=dff['ridership'],
        name=f"{selected_mode} Daily Ridership",
        yaxis="y1"
    ))
    fig.add_trace(go.Scatter(
        x=dff['year'],
        y=dff['ave_distance_per_trip'],
        mode="lines+markers",
        name=f"{selected_mode} Avg Trip Distance",
        yaxis="y2"
    ))
    fig.update_layout(
        title=f"Daily Ridership & Avg Trip Distance for {selected_mode}",
        xaxis=dict(title="Year"),
        yaxis=dict(title="Daily Ridership", side="left"),
        yaxis2=dict(title="Avg Trip Distance (km)", overlaying="y", side="right"),
        template="plotly_dark"
    )
    total_ridership = dff['ridership'].sum()
    avg_trip_distance = dff['ave_distance_per_trip'].mean()
    info_cards = dbc.Row([
        dbc.Col(dbc.Card([
            dbc.CardHeader("Total Ridership"),
            dbc.CardBody(html.H4(f"{total_ridership:,.0f}", className="card-text"))
        ], color="info", inverse=True), md=6),
        dbc.Col(dbc.Card([
            dbc.CardHeader("Average Trip Distance (km)"),
            dbc.CardBody(html.H4(f"{avg_trip_distance:.2f}", className="card-text"))
        ], color="primary", inverse=True), md=6)
    ], className="mb-3")
    return fig, info_cards

@app.callback(
    [Output('box-plot-chart', 'figure'),
     Output('box-plot-info-cards', 'children')],
    [Input('time-dropdown', 'value')]
)
def update_box_plot(selected_times):
    if selected_times:
        dff = df_fares[df_fares['applicable_time'].isin(selected_times)]
    else:
        dff = df_fares.copy()
    fig = px.box(
        dff,
        x="distance_bin",
        y="cost_per_km",
        color="applicable_time",
        points="all",
        title="Cost per KM Distribution by Distance Bin & Applicable Time",
        template="plotly_dark"
    )
    fig.update_layout(
        xaxis_title="Trip Distance Range (km)",
        yaxis_title="Fare Cost per Kilometer"
    )
    fig.update_xaxes(
        tickmode='array',
        tickvals=["(0,3.2]", "(3.2,7]", "(7,12]", "(12,inf)"],
        ticktext=["0 to 3.2 km", "3.2 to 7 km", "7 to 12 km", "12+ km"]
    )
    avg_cost_per_km = dff['cost_per_km'].mean()
    median_cost_per_km = dff['cost_per_km'].median()
    num_records = dff.shape[0]
    
    
    # Create Cards
    info_cards = dbc.Row([
        dbc.Col(dbc.Card([
            dbc.CardHeader("Avg Fare Cost per KM"),
            dbc.CardBody(html.H4(f"{avg_cost_per_km:.2f}", className="card-text"))
        ], color="warning", inverse=True), md=4),
        dbc.Col(dbc.Card([
            dbc.CardHeader("Median Fare Cost per KM"),
            dbc.CardBody(html.H4(f"{median_cost_per_km:.2f}", className="card-text"))
        ], color="secondary", inverse=True), md=4),
        dbc.Col(dbc.Card([
            dbc.CardHeader("Data Points"),
            dbc.CardBody(html.H4(f"{num_records}", className="card-text"))
        ], color="danger", inverse=True), md=4)
    ], className="mb-3")
    
    return fig, info_cards

if __name__ == '__main__':
    app.run_server(debug=True, port=8050)
